# ML Notebook: Apache Iceberg Offline Store To Saved Model

This notebook follows the rubric steps directly with the current data platform design:

1. Load BST-ready training samples from the cluster Apache Iceberg offline feature table `recsys_features.feature_store.ml_bst_training`.
2. Export temporal train/validation/test JSONL splits compatible with `recommenderDataset`.
3. Train the existing BST binary classification model on the training split.
4. Evaluate the model on the validation split.
5. Save the trained model and Iceberg data lineage as `.joblib`.

Note: the cluster may still expose an S3-compatible endpoint as the storage backend for the Iceberg warehouse, but the offline feature store abstraction used here is Apache Iceberg, not raw object-store parquet files.

## Setup: paths and imports

In [3]:
from __future__ import annotations

import json
import os
import random
import shutil
import subprocess
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "apps/ml-system").exists():
            return candidate
    raise RuntimeError("Could not find RecSys-MLops repository root")


def run_command(command: list[str], *, check: bool = True) -> subprocess.CompletedProcess[str]:
    printable = " ".join(command)
    print(f"$ {printable}")
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed with exit code {result.returncode}: {printable}")
    return result



def read_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


ROOT = find_repo_root(Path.cwd().resolve())
ML_SYSTEM_SRC = ROOT / "apps/ml-system/src"
sys.path.insert(0, str(ML_SYSTEM_SRC))

NAMESPACE = os.getenv("DATA_PLATFORM_NAMESPACE", "recsys-dataflow")
EXPORT_POD = os.getenv("ICEBERG_EXPORT_POD", "recsys-iceberg-training-export")


def _derive_mlops_spark_image(spark_image: str) -> str:
    return spark_image.replace("/recsys-spark:", "/recsys-mlops-spark:").replace("recsys-spark:", "recsys-mlops-spark:")


def resolve_export_image() -> str:
    configured = os.getenv("RECSYS_SPARK_IMAGE", "").strip()
    if configured and not configured.endswith(":local"):
        return configured

    result = run_command(
        [
            "kubectl",
            "get",
            "configmap",
            "recsys-data-platform-config",
            "-n",
            NAMESPACE,
            "-o",
            "jsonpath={.data.SPARK_IMAGE}",
        ],
        check=False,
    )
    cluster_spark_image = result.stdout.strip()
    if cluster_spark_image:
        return _derive_mlops_spark_image(cluster_spark_image)

    return configured or "recsys-mlops-spark:local"


EXPORT_IMAGE = resolve_export_image()
SPARK_PACKAGES = os.getenv(
    "RECSYS_SPARK_PACKAGES",
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.9.2,"
    "org.apache.hudi:hudi-spark3.5-bundle_2.12:1.0.2,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262",
)

ICEBERG_CATALOG_NAME = os.getenv("ICEBERG_CATALOG_NAME", "recsys_features")
ICEBERG_WAREHOUSE = os.getenv("ICEBERG_WAREHOUSE", "s3a://recsys-offline-feature-store/warehouse")
OFFLINE_FEATURE_TABLE = os.getenv("OFFLINE_FEATURE_TABLE", "recsys_features.feature_store.ml_bst_training")
FEATURE_SERVICE_NAME = os.getenv("FEATURE_SERVICE_NAME", "bst_ranking_v1")
MAX_HISTORY_LEN = int(os.getenv("NOTEBOOK_MAX_HISTORY_LEN", "5"))
RANDOM_SEED = int(os.getenv("NOTEBOOK_RANDOM_SEED", "42"))
TRAINING_PERCENT = float(os.getenv("NOTEBOOK_TRAINING_PERCENT", "1.0"))

BST_SPLIT_DIR = ROOT / "notebooks/data/iceberg_bst_split"
MODEL_DIR = ROOT / "notebooks/models"
CHECKPOINT_DIR = MODEL_DIR / "iceberg_bst_checkpoints"
MODEL_PATH = MODEL_DIR / "iceberg_bst_10epoch.joblib"
DATASET_METADATA_PATH = BST_SPLIT_DIR / "dataset_version_meta.json"
SPLIT_META_PATH = BST_SPLIT_DIR / "split_meta.json"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
BST_SPLIT_DIR.parent.mkdir(parents=True, exist_ok=True)

print(f"repo root: {ROOT}")
print(f"K8s namespace: {NAMESPACE}")
print(f"Spark/Iceberg export pod image: {EXPORT_IMAGE}")
print(f"offline feature table: {OFFLINE_FEATURE_TABLE}")
print(f"Iceberg catalog: {ICEBERG_CATALOG_NAME}")
print(f"Iceberg warehouse: {ICEBERG_WAREHOUSE}")
print(f"ML system src: {ML_SYSTEM_SRC.relative_to(ROOT)}")

$ kubectl get configmap recsys-data-platform-config -n recsys-dataflow -o jsonpath={.data.SPARK_IMAGE}
asia-southeast1-docker.pkg.dev/fsds-coursework/recsys/recsys-spark:gcp
repo root: /Users/KHOAI/anhkhoa/RecSys-MLops
K8s namespace: recsys-dataflow
Spark/Iceberg export pod image: asia-southeast1-docker.pkg.dev/fsds-coursework/recsys/recsys-mlops-spark:gcp
offline feature table: recsys_features.feature_store.ml_bst_training
Iceberg catalog: recsys_features
Iceberg warehouse: s3a://recsys-offline-feature-store/warehouse
ML system src: apps/ml-system/src


## Step 1. Load data from Apache Iceberg offline feature store

The current data platform writes the BST training table to Apache Iceberg at `recsys_features.feature_store.ml_bst_training`. This table is produced by Spark batch feature engineering and already contains point-in-time training labels plus the BST sequence, target item, target category, target brand, target price bucket, event time, and label columns.

This step starts a temporary Spark pod in the cluster, reads the Iceberg table, exports temporal train/validation/test JSONL splits through the existing `prepare_bst_training_data.py` helper, and copies those splits back to this notebook.

### Step 1a. Export BST splits from the Iceberg offline feature table

In [4]:
shutil.rmtree(BST_SPLIT_DIR, ignore_errors=True)

run_command(["kubectl", "delete", "pod", "-n", NAMESPACE, EXPORT_POD, "--ignore-not-found=true"])
pod_overrides = {
    "metadata": {
        "annotations": {
            "sidecar.istio.io/inject": "false",
        }
    },
    "spec": {
        "containers": [
            {
                "name": EXPORT_POD,
                "image": EXPORT_IMAGE,
                "imagePullPolicy": "IfNotPresent",
                "command": ["sleep"],
                "args": ["3600"],
                "envFrom": [
                    {"configMapRef": {"name": "recsys-data-platform-config"}},
                    {"secretRef": {"name": "recsys-data-platform-secret"}},
                ],
                "env": [
                    {"name": "PYTHONPATH", "value": "/opt/recsys/apps/ml-system/src:/opt/recsys"},
                    {"name": "AWS_DEFAULT_REGION", "value": "us-east-1"},
                    {"name": "MINIO_ENDPOINT", "value": "http://data-platform-minio:9000"},
                    {"name": "AWS_ENDPOINT_URL", "value": "http://data-platform-minio:9000"},
                    {"name": "S3_ENDPOINT_URL", "value": "http://data-platform-minio:9000"},
                    {"name": "ICEBERG_CATALOG_NAME", "value": ICEBERG_CATALOG_NAME},
                    {"name": "ICEBERG_WAREHOUSE", "value": ICEBERG_WAREHOUSE},
                    {"name": "OFFLINE_FEATURE_TABLE", "value": OFFLINE_FEATURE_TABLE},
                    {"name": "AWS_ALLOW_HTTP", "value": "true"},
                    {"name": "AWS_EC2_METADATA_DISABLED", "value": "true"},
                ],
            }
        ]
    }
}
run_command([
    "kubectl",
    "run",
    EXPORT_POD,
    "-n",
    NAMESPACE,
    "--image",
    EXPORT_IMAGE,
    "--image-pull-policy=IfNotPresent",
    "--restart=Never",
    "--overrides",
    json.dumps(pod_overrides),
])
wait_result = run_command(
    ["kubectl", "wait", "-n", NAMESPACE, "--for=condition=Ready", f"pod/{EXPORT_POD}", "--timeout=180s"],
    check=False,
)
if wait_result.returncode != 0:
    run_command(["kubectl", "get", "pod", EXPORT_POD, "-n", NAMESPACE, "-o", "wide"], check=False)
    run_command(["kubectl", "describe", "pod", EXPORT_POD, "-n", NAMESPACE], check=False)
    raise RuntimeError(
        f"Export pod did not become Ready. Check that EXPORT_IMAGE is pullable: {EXPORT_IMAGE}"
    )

spark_submit = [
    "/opt/spark/bin/spark-submit",
    "--master",
    "local[*]",
    "--packages",
    SPARK_PACKAGES,
    "/opt/recsys/apps/ml-system/src/cli/prepare_bst_training_data.py",
    "--feature-source",
    "offline_feature_store",
    "--offline-feature-table",
    OFFLINE_FEATURE_TABLE,
    "--output-dir",
    "/tmp/bst_split",
    "--train-ratio",
    "0.8",
    "--val-ratio",
    "0.1",
    "--max-history-len",
    str(MAX_HISTORY_LEN),
    "--feature-service-name",
    FEATURE_SERVICE_NAME,
    "--hudi-enabled",
    "false",
    "--iceberg-enabled",
    "false",
    "--iceberg-catalog-name",
    ICEBERG_CATALOG_NAME,
    "--iceberg-warehouse",
    ICEBERG_WAREHOUSE,
    "--dataset-run-id",
    "notebook-iceberg-bst",
    "--dataset-metadata-path",
    "/tmp/bst_split/dataset_version_meta.json",
]
run_command(["kubectl", "exec", "-n", NAMESPACE, EXPORT_POD, "--", *spark_submit])
run_command(["kubectl", "cp", f"{NAMESPACE}/{EXPORT_POD}:/tmp/bst_split", str(BST_SPLIT_DIR)])

print(f"Iceberg BST split copied to: {BST_SPLIT_DIR.relative_to(ROOT)}")

$ kubectl delete pod -n recsys-dataflow recsys-iceberg-training-export --ignore-not-found=true
pod "recsys-iceberg-training-export" deleted from recsys-dataflow namespace

$ kubectl run recsys-iceberg-training-export -n recsys-dataflow --image asia-southeast1-docker.pkg.dev/fsds-coursework/recsys/recsys-mlops-spark:gcp --image-pull-policy=IfNotPresent --restart=Never --overrides {"metadata": {"annotations": {"sidecar.istio.io/inject": "false"}}, "spec": {"containers": [{"name": "recsys-iceberg-training-export", "image": "asia-southeast1-docker.pkg.dev/fsds-coursework/recsys/recsys-mlops-spark:gcp", "imagePullPolicy": "IfNotPresent", "command": ["sleep"], "args": ["3600"], "envFrom": [{"configMapRef": {"name": "recsys-data-platform-config"}}, {"secretRef": {"name": "recsys-data-platform-secret"}}], "env": [{"name": "PYTHONPATH", "value": "/opt/recsys/apps/ml-system/src:/opt/recsys"}, {"name": "AWS_DEFAULT_REGION", "value": "us-east-1"}, {"name": "MINIO_ENDPOINT", "value": "http://data-p

### Step 1b. Load the exported Iceberg training splits locally

In [ ]:
train_path = BST_SPLIT_DIR / "train.jsonl"
val_path = BST_SPLIT_DIR / "val.jsonl"
test_path = BST_SPLIT_DIR / "test.jsonl"

train_records = read_jsonl(train_path)
val_records = read_jsonl(val_path)
test_records = read_jsonl(test_path)
split_meta = json.loads(SPLIT_META_PATH.read_text())
dataset_metadata = json.loads(DATASET_METADATA_PATH.read_text())

BST_MODEL_COLUMNS = [
    "user_id",
    "hist_item_id",
    "hist_event_type",
    "hist_category",
    "hist_brand",
    "hist_price_bucket",
    "hist_time",
    "target_item_id",
    "target_category",
    "target_brand",
    "target_price_bucket",
    "event_time",
    "label",
]

label_summary = {
    split: pd.Series([record["label"] for record in records]).value_counts().sort_index().to_dict()
    for split, records in {"train": train_records, "val": val_records, "test": test_records}.items()
}

print(f"offline feature table: {split_meta['offline_feature_table']}")
print(f"feature source: {split_meta['feature_source']}")
print(f"total rows: {split_meta['total_rows']:,}")
print(f"train/val/test rows: {len(train_records):,} / {len(val_records):,} / {len(test_records):,}")
print("label distribution:", label_summary)
print("schema hash:", split_meta["schema_hash"])
print("dataset metadata:", DATASET_METADATA_PATH.relative_to(ROOT))
print("sample Iceberg-derived recommenderDataset JSONL record:")
print(json.dumps(train_records[0], indent=2)[:1200])

## Step 2. Use the Iceberg-derived train/validation/test split

`prepare_bst_training_data.py` sorts the Iceberg table by `prediction_timestamp` when available, otherwise by `event_time`, then writes temporal splits. This keeps the notebook aligned with the production Kubeflow training pipeline and avoids synthetic labels/features in notebook code.

In [ ]:
ITEM_NUM = 22_700
CATEGORY_NUM = 30
BRAND_NUM = 740
PRICE_BUCKET_NUM = 10
TIME_BUCKET_NUM = 9
EVENT_TYPE_NUM = 3

MODEL_CONFIG = {
    "model_args": {
        "n_heads": 2,
        "k_interests": 4,
        "embed_dim": 8,
        "seq_len": MAX_HISTORY_LEN + 1,
        "intermediate_size": 32,
        "hidden_dropout_prob": 0.1,
        "attn_dropout_prob": 0.1,
        "hidden_act": "relu",
        "layer_norm_eps": 1e-12,
        "padding_idx": 0,
        "reload_model": False,
        "save_path": str(CHECKPOINT_DIR),
        "item_num": ITEM_NUM,
        "category_num": CATEGORY_NUM,
        "brand_num": BRAND_NUM,
        "price_bucket_num": PRICE_BUCKET_NUM,
        "time_bucket_num": TIME_BUCKET_NUM,
        "event_type_num": EVENT_TYPE_NUM,
    },
    "data_args": {
        "num_workers": 0,
        "shuffle": True,
        "max_history_len": MAX_HISTORY_LEN,
        "train_data_path": str(train_path),
        "val_data_path": str(val_path),
        "test_data_path": str(test_path),
        "padding_idx": 0,
    },
    "training_args": {
        "num_epochs": 10,
        "learning_rate": 0.001,
        "weight_decay": 0.00001,
        "batch_size": 256,
        "num_workers": 0,
        "ranking_ks": [1, 3, 5, 10],
    },
}

print(f"train JSONL: {train_path.relative_to(ROOT)}")
print(f"validation JSONL: {val_path.relative_to(ROOT)}")
print(f"test JSONL: {test_path.relative_to(ROOT)}")
print(f"max history length: {MAX_HISTORY_LEN}")
print(f"training percent: {TRAINING_PERCENT}")

## Step 3. Train model on the Iceberg-derived training dataset

This step uses the existing `recommenderDataset`, `Trainer`, and `BST` classes from `apps/ml-system/src/models`. Training is capped at 10 epochs.

In [ ]:
import torch

from models.dataset import recommenderDataset
from models.trainer import Trainer


torch.manual_seed(RANDOM_SEED)
trainer = Trainer(MODEL_CONFIG)
train_dataset = recommenderDataset(MODEL_CONFIG["data_args"], split="train", percent=TRAINING_PERCENT)
val_dataset = recommenderDataset(MODEL_CONFIG["data_args"], split="val", percent=1.0)
train_loader = trainer.get_data_loader(train_dataset, shuffle=True)
val_loader = trainer.get_data_loader(val_dataset, shuffle=False)

print(f"dataset class: {train_dataset.__class__.__name__}")
print(f"model class: {trainer.model.__class__.__name__}")
print(f"trainer class: {trainer.__class__.__name__}")
print(f"training device: {trainer.device}")
print(f"epochs: {MODEL_CONFIG['training_args']['num_epochs']}")
print(f"training rows used: {len(train_dataset):,}")

history = []
for epoch in range(1, MODEL_CONFIG["training_args"]["num_epochs"] + 1):
    train_metrics = trainer.train(train_loader)
    val_metrics = trainer.evaluate(val_loader)
    trainer.scheduler.step(val_metrics["loss"])
    history.append({"epoch": epoch, "train": train_metrics, "val": val_metrics})
    print(
        f"epoch={epoch:02d} "
        f"train_loss={train_metrics['loss']:.4f} train_auc={train_metrics['auc']:.4f} "
        f"val_loss={val_metrics['loss']:.4f} val_auc={val_metrics['auc']:.4f} "
        f"val_ndcg@10={val_metrics.get('ndcg@10', 0.0):.4f}"
    )

model = trainer.model
metrics = {
    "epochs": MODEL_CONFIG["training_args"]["num_epochs"],
    "final_train": history[-1]["train"],
    "final_validation": history[-1]["val"],
    "history": history,
}

## Step 4. Evaluate on validation dataset

In [ ]:
validation_metrics = metrics["final_validation"]

print("validation metrics from apps/ml-system Trainer.evaluate:")
for key in sorted(validation_metrics):
    value = validation_metrics[key]
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

## Step 5. Save model as `.joblib`

In [ ]:
artifact = {
    "model_type": "BST",
    "dataset_class": "recommenderDataset",
    "trainer_class": "Trainer",
    "task": "binary_classification",
    "offline_store_type": "apache_iceberg",
    "offline_feature_table": OFFLINE_FEATURE_TABLE,
    "iceberg_catalog_name": ICEBERG_CATALOG_NAME,
    "iceberg_warehouse": ICEBERG_WAREHOUSE,
    "feature_service_name": FEATURE_SERVICE_NAME,
    "feature_columns": BST_MODEL_COLUMNS,
    "state_dict": {key: value.detach().cpu().numpy() for key, value in model.state_dict().items()},
    "model_config": MODEL_CONFIG,
    "metrics": metrics,
    "trained_device": str(trainer.device),
    "cluster_namespace": NAMESPACE,
    "cluster_export_pod": EXPORT_POD,
    "training_dataset_dir": str(BST_SPLIT_DIR.relative_to(ROOT)),
    "dataset_metadata": dataset_metadata,
    "split_metadata": split_meta,
    "train_jsonl": str(train_path.relative_to(ROOT)),
    "val_jsonl": str(val_path.relative_to(ROOT)),
    "test_jsonl": str(test_path.relative_to(ROOT)),
    "random_seed": RANDOM_SEED,
}
joblib.dump(artifact, MODEL_PATH)

print(f"saved model artifact: {MODEL_PATH.relative_to(ROOT)}")
print(f"artifact size bytes: {MODEL_PATH.stat().st_size:,}")
print("saved artifact keys:", sorted(artifact.keys()))

## Document main steps completed in this notebook

- **Step 1 - Load data from Apache Iceberg offline feature store:** Started a temporary Spark pod, read `recsys_features.feature_store.ml_bst_training`, and exported BST JSONL splits with `prepare_bst_training_data.py`.
- **Step 2 - Split data:** Used the production temporal split logic from `prepare_bst_training_data.py` to create train, validation, and test JSONL files compatible with `recommenderDataset`.
- **Step 3 - Train model:** Trained the existing `BST` model through the existing `Trainer` class for exactly 10 epochs on `mps` when available, otherwise CPU.
- **Step 4 - Evaluate model:** Reported validation metrics from `Trainer.evaluate`, including AUC and ranking metrics such as NDCG/MRR/hitrate.
- **Step 5 - Save model:** Saved the BST model state, config, metrics, dataset paths, Iceberg table name, Iceberg warehouse, and data lineage to `notebooks/models/iceberg_bst_10epoch.joblib`.